# Notebook to prepare adult data for a unique file
## protected attributes are sex, race, age
## for this pre-processing we are binarizing the protected attributes

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from falsb4mpa.dataset.utils import bucket

In [2]:
column_names = ['age',
'workclass', 
'fnlwgt', 
'education',
'education-num',
'marital-status',
'occupation',
'relationship',
'race',
'sex',
'capital-gain',
'capital-loss',
'hours-per-week',
'native-country',
'income']

In [3]:
used_columns = ['age',
'workclass',
'education',
'education-num',
'marital-status',
'occupation',
'relationship',
'race',
'capital-gain',
'capital-loss',
'hours-per-week',
'native-country',]
target = 'income'
sensitive = 'sex'

# Reading data

In [4]:
adult_train_data = pd.read_csv("../../data/raw/adult/adult.data", header=None, names=column_names)
adult_test_data = pd.read_csv("../../data/raw/adult/adult.test", header=None, names=column_names)

In [5]:
adult_train_data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [6]:
adult_train_data.tail()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K
32560,52,Self-emp-inc,287927,HS-grad,9,Married-civ-spouse,Exec-managerial,Wife,White,Female,15024,0,40,United-States,>50K


In [7]:
adult_test_data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,|1x3 Cross validator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25,Private,226802.0,11th,7.0,Never-married,Machine-op-inspct,Own-child,Black,Male,0.0,0.0,40.0,United-States,<=50K.
2,38,Private,89814.0,HS-grad,9.0,Married-civ-spouse,Farming-fishing,Husband,White,Male,0.0,0.0,50.0,United-States,<=50K.
3,28,Local-gov,336951.0,Assoc-acdm,12.0,Married-civ-spouse,Protective-serv,Husband,White,Male,0.0,0.0,40.0,United-States,>50K.
4,44,Private,160323.0,Some-college,10.0,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688.0,0.0,40.0,United-States,>50K.


In [8]:
adult_test_data.drop(labels=[0], axis=0, inplace=True)
adult_test_data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
1,25,Private,226802.0,11th,7.0,Never-married,Machine-op-inspct,Own-child,Black,Male,0.0,0.0,40.0,United-States,<=50K.
2,38,Private,89814.0,HS-grad,9.0,Married-civ-spouse,Farming-fishing,Husband,White,Male,0.0,0.0,50.0,United-States,<=50K.
3,28,Local-gov,336951.0,Assoc-acdm,12.0,Married-civ-spouse,Protective-serv,Husband,White,Male,0.0,0.0,40.0,United-States,>50K.
4,44,Private,160323.0,Some-college,10.0,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688.0,0.0,40.0,United-States,>50K.
5,18,?,103497.0,Some-college,10.0,Never-married,?,Own-child,White,Female,0.0,0.0,30.0,United-States,<=50K.


In [9]:
adult_test_data.tail()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
16277,39,Private,215419.0,Bachelors,13.0,Divorced,Prof-specialty,Not-in-family,White,Female,0.0,0.0,36.0,United-States,<=50K.
16278,64,?,321403.0,HS-grad,9.0,Widowed,?,Other-relative,Black,Male,0.0,0.0,40.0,United-States,<=50K.
16279,38,Private,374983.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Husband,White,Male,0.0,0.0,50.0,United-States,<=50K.
16280,44,Private,83891.0,Bachelors,13.0,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455.0,0.0,40.0,United-States,<=50K.
16281,35,Self-emp-inc,182148.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,60.0,United-States,>50K.


In [10]:
print(len(adult_train_data.index))
print(len(adult_test_data.index))

32561
16281


# Removing missing values

In [11]:
adult_test_data['workclass'].iloc[4] == (' ?')

True

In [12]:
column_names = ['age',
    'workclass', 
    'fnlwgt', 
    'education',
    'education-num',
    'marital-status',
    'occupation',
    'relationship',
    'capital-gain',
    'capital-loss',
    'hours-per-week',
    'native-country',
    'income',
    'sex',
    'race'
]

In [13]:
for column in column_names:
    #print(column)
    #print(adult_train_data[column])
    adult_train_data = adult_train_data[adult_train_data[column] != ' ?']
    adult_test_data = adult_test_data[adult_test_data[column] != ' ?']

In [14]:
print(len(adult_train_data.index))
print(len(adult_test_data.index))

30162
15060


# Normalizing continuous data

In [15]:
continous_attr = ['capital-gain', 'capital-loss', 'hours-per-week']
scaler = MinMaxScaler()

In [16]:
for attr in continous_attr:
    adult_train_data[attr] = scaler.fit_transform(np.array(adult_train_data[attr]).reshape(-1,1))
    adult_test_data[attr] = scaler.fit_transform(np.array(adult_test_data[attr]).reshape(-1,1))

In [17]:
adult_train_data.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,30162.000000,3.016200e+04,30162.000000,30162.000000,30162.000000,30162.000000
mean,38.437902,1.897938e+05,10.121312,0.010920,0.020288,0.407462
std,13.134665,1.056530e+05,2.549995,0.074064,0.092814,0.122245
min,17.000000,1.376900e+04,1.000000,0.000000,0.000000,0.000000
25%,28.000000,1.176272e+05,9.000000,0.000000,0.000000,0.397959
50%,37.000000,1.784250e+05,10.000000,0.000000,0.000000,0.397959
75%,47.000000,2.376285e+05,13.000000,0.000000,0.000000,0.448980
max,90.000000,1.484705e+06,16.000000,1.000000,1.000000,1.000000


In [18]:
adult_test_data.describe()

,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,1.506000e+04,15060.000000,15060.000000,15060.000000,15060.000000
mean,1.896164e+05,10.112749,0.011203,0.023619,0.407669
std,1.056150e+05,2.558727,0.077033,0.107767,0.123090
min,1.349200e+04,1.000000,0.000000,0.000000,0.000000
25%,1.166550e+05,9.000000,0.000000,0.000000,0.397959
50%,1.779550e+05,10.000000,0.000000,0.000000,0.397959
75%,2.385888e+05,13.000000,0.000000,0.000000,0.448980
max,1.490400e+06,16.000000,1.000000,1.000000,1.000000


# Binarizing age

In [19]:
adult_train_data['age'].describe()

count    30162.000000
mean        38.437902
std         13.134665
min         17.000000
25%         28.000000
50%         37.000000
75%         47.000000
max         90.000000
Name: age, dtype: float64

In [20]:
adult_train_data['age'].head()

0    39
1    50
2    38
3    53
4    28
Name: age, dtype: int64

In [21]:
adult_train_data['age'] = adult_train_data['age'].between(30, 60, inclusive='both').astype(int)
adult_train_data['age'].head()

0    1
1    1
2    1
3    1
4    0
Name: age, dtype: int64

In [22]:
adult_test_data['age'] = adult_test_data['age'].astype(int)
adult_test_data['age'].head()

1    25
2    38
3    28
4    44
6    34
Name: age, dtype: int64

In [23]:
adult_test_data['age'] = adult_test_data['age'].between(30, 60, inclusive='both').astype(int)
adult_test_data['age'].head()

1    0
2    1
3    0
4    1
6    1
Name: age, dtype: int64

# Binarizing race and sex

In [24]:
adult_train_data['sex'] = pd.get_dummies(adult_train_data['sex']).astype(int)[' Male']
adult_test_data['sex'] = pd.get_dummies(adult_test_data['sex']).astype(int)[' Male']

In [25]:
adult_train_data['race'] = pd.get_dummies(adult_train_data['race']).astype(int)[' White']
adult_test_data['race'] = pd.get_dummies(adult_test_data['race']).astype(int)[' White']

# Binarizing income

In [26]:
adult_train_data['income'] = pd.get_dummies(adult_train_data['income']).astype(int)[' >50K']
adult_test_data['income'] = pd.get_dummies(adult_test_data['income']).astype(int)[' >50K.']

# One hot encoding categorical data

In [27]:
categorical_attr = ['workclass','education','marital-status','occupation','relationship','native-country']

In [28]:
one_hot_train = adult_train_data.copy()
one_hot_test = adult_test_data.copy()

for attr in categorical_attr:
    column_idx = adult_train_data.columns.get_loc(attr)
    adult_train_data = pd.concat([adult_train_data, pd.get_dummies(adult_train_data[attr], prefix=attr, dtype=int)], axis=1)
    adult_test_data = pd.concat([adult_test_data, pd.get_dummies(adult_test_data[attr], prefix=attr, dtype=int)], axis=1)

In [29]:
adult_train_data

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,...,native-country_ Portugal,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia
0,1,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,1,1,...,0,0,0,0,0,0,0,1,0,0
1,1,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,1,1,...,0,0,0,0,0,0,0,1,0,0
2,1,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,1,1,...,0,0,0,0,0,0,0,1,0,0
3,1,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,0,1,...,0,0,0,0,0,0,0,1,0,0
4,0,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,0,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,1,0,...,0,0,0,0,0,0,0,1,0,0
32557,1,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,1,1,...,0,0,0,0,0,0,0,1,0,0
32558,1,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,1,0,...,0,0,0,0,0,0,0,1,0,0
32559,0,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,1,1,...,0,0,0,0,0,0,0,1,0,0


In [30]:
adult_test_data

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,...,native-country_ Portugal,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia
1,0,Private,226802.0,11th,7.0,Never-married,Machine-op-inspct,Own-child,0,1,...,0,0,0,0,0,0,0,1,0,0
2,1,Private,89814.0,HS-grad,9.0,Married-civ-spouse,Farming-fishing,Husband,1,1,...,0,0,0,0,0,0,0,1,0,0
3,0,Local-gov,336951.0,Assoc-acdm,12.0,Married-civ-spouse,Protective-serv,Husband,1,1,...,0,0,0,0,0,0,0,1,0,0
4,1,Private,160323.0,Some-college,10.0,Married-civ-spouse,Machine-op-inspct,Husband,0,1,...,0,0,0,0,0,0,0,1,0,0
6,1,Private,198693.0,10th,6.0,Never-married,Other-service,Not-in-family,1,1,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16276,1,Private,245211.0,Bachelors,13.0,Never-married,Prof-specialty,Own-child,1,1,...,0,0,0,0,0,0,0,1,0,0
16277,1,Private,215419.0,Bachelors,13.0,Divorced,Prof-specialty,Not-in-family,1,0,...,0,0,0,0,0,0,0,1,0,0
16279,1,Private,374983.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Husband,1,1,...,0,0,0,0,0,0,0,1,0,0
16280,1,Private,83891.0,Bachelors,13.0,Divorced,Adm-clerical,Own-child,0,1,...,0,0,0,0,0,0,0,1,0,0


# Removing not used columns

In [31]:
attr_to_remove = ['fnlwgt'] + categorical_attr

In [32]:
for attr in adult_train_data.columns:
    if '_ ?' in attr:
        print(attr)
        adult_train_data = adult_train_data.drop(attr, axis='columns')
        adult_test_data = adult_test_data.drop(attr, axis='columns')

In [33]:
print(len(adult_test_data.index))

15060


In [34]:
for attr in attr_to_remove:
    adult_train_data = adult_train_data.drop(attr, axis='columns')
    adult_test_data = adult_test_data.drop(attr, axis='columns')

In [35]:
print(len(adult_test_data.index))

15060


In [36]:
print(len(adult_train_data.columns))
print(len(adult_test_data.columns))

99
98


In [37]:
#Test file doesn't have any example for this value of native-country -> get_dummies doesn't encode it
attr = 'native-country_ Holand-Netherlands'
rows = len(adult_test_data.index)
d = pd.DataFrame(0, index=np.arange(rows), columns=[attr])
d

,native-country_ Holand-Netherlands
0,0
1,0
2,0
3,0
4,0
...,...
15055,0
15056,0
15057,0
15058,0


In [38]:
adult_test_data

,age,education-num,race,sex,capital-gain,capital-loss,hours-per-week,income,workclass_ Federal-gov,workclass_ Local-gov,...,native-country_ Portugal,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia
1,0,7.0,0,1,0.000000,0.0,0.397959,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,1,9.0,1,1,0.000000,0.0,0.500000,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0,12.0,1,1,0.000000,0.0,0.397959,1,0,1,...,0,0,0,0,0,0,0,1,0,0
4,1,10.0,0,1,0.076881,0.0,0.397959,1,0,0,...,0,0,0,0,0,0,0,1,0,0
6,1,6.0,1,1,0.000000,0.0,0.295918,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16276,1,13.0,1,1,0.000000,0.0,0.397959,0,0,0,...,0,0,0,0,0,0,0,1,0,0
16277,1,13.0,1,0,0.000000,0.0,0.357143,0,0,0,...,0,0,0,0,0,0,0,1,0,0
16279,1,13.0,1,1,0.000000,0.0,0.500000,0,0,0,...,0,0,0,0,0,0,0,1,0,0
16280,1,13.0,0,1,0.054551,0.0,0.397959,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [39]:
adult_test_data.reset_index(drop=True, inplace=True)
d.reset_index(drop=True, inplace=True)
pd.concat([adult_test_data, d], axis=1)

,age,education-num,race,sex,capital-gain,capital-loss,hours-per-week,income,workclass_ Federal-gov,workclass_ Local-gov,...,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia,native-country_ Holand-Netherlands
0,0,7.0,0,1,0.000000,0.0,0.397959,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,1,9.0,1,1,0.000000,0.0,0.500000,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,0,12.0,1,1,0.000000,0.0,0.397959,1,0,1,...,0,0,0,0,0,0,1,0,0,0
3,1,10.0,0,1,0.076881,0.0,0.397959,1,0,0,...,0,0,0,0,0,0,1,0,0,0
4,1,6.0,1,1,0.000000,0.0,0.295918,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15055,1,13.0,1,1,0.000000,0.0,0.397959,0,0,0,...,0,0,0,0,0,0,1,0,0,0
15056,1,13.0,1,0,0.000000,0.0,0.357143,0,0,0,...,0,0,0,0,0,0,1,0,0,0
15057,1,13.0,1,1,0.000000,0.0,0.500000,0,0,0,...,0,0,0,0,0,0,1,0,0,0
15058,1,13.0,0,1,0.054551,0.0,0.397959,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [40]:
adult_test_data = pd.concat([adult_test_data, d], axis=1)
print(len(adult_test_data.columns))

99


In [41]:
adult_train_data.columns.to_list()

['age',
 'education-num',
 'race',
 'sex',
 'capital-gain',
 'capital-loss',
 'hours-per-week',
 'income',
 'workclass_ Federal-gov',
 'workclass_ Local-gov',
 'workclass_ Private',
 'workclass_ Self-emp-inc',
 'workclass_ Self-emp-not-inc',
 'workclass_ State-gov',
 'workclass_ Without-pay',
 'education_ 10th',
 'education_ 11th',
 'education_ 12th',
 'education_ 1st-4th',
 'education_ 5th-6th',
 'education_ 7th-8th',
 'education_ 9th',
 'education_ Assoc-acdm',
 'education_ Assoc-voc',
 'education_ Bachelors',
 'education_ Doctorate',
 'education_ HS-grad',
 'education_ Masters',
 'education_ Preschool',
 'education_ Prof-school',
 'education_ Some-college',
 'marital-status_ Divorced',
 'marital-status_ Married-AF-spouse',
 'marital-status_ Married-civ-spouse',
 'marital-status_ Married-spouse-absent',
 'marital-status_ Never-married',
 'marital-status_ Separated',
 'marital-status_ Widowed',
 'occupation_ Adm-clerical',
 'occupation_ Armed-Forces',
 'occupation_ Craft-repair',
 'oc

# Reordering the columns

In [42]:
#sem education-num
columns_order = [
    'age',
    'sex',
    'race',
    'education_ 10th','education_ 11th','education_ 12th','education_ 1st-4th','education_ 5th-6th','education_ 7th-8th','education_ 9th','education_ Assoc-acdm','education_ Assoc-voc','education_ Bachelors','education_ Doctorate','education_ HS-grad','education_ Masters','education_ Preschool','education_ Prof-school','education_ Some-college',
    'workclass_ Federal-gov','workclass_ Local-gov','workclass_ Private','workclass_ Self-emp-inc','workclass_ Self-emp-not-inc','workclass_ State-gov','workclass_ Without-pay',
    'marital-status_ Divorced','marital-status_ Married-AF-spouse','marital-status_ Married-civ-spouse','marital-status_ Married-spouse-absent','marital-status_ Never-married','marital-status_ Separated','marital-status_ Widowed',
    'occupation_ Adm-clerical','occupation_ Armed-Forces','occupation_ Craft-repair','occupation_ Exec-managerial','occupation_ Farming-fishing','occupation_ Handlers-cleaners','occupation_ Machine-op-inspct','occupation_ Other-service','occupation_ Priv-house-serv','occupation_ Prof-specialty','occupation_ Protective-serv','occupation_ Sales','occupation_ Tech-support','occupation_ Transport-moving',
    'relationship_ Husband','relationship_ Not-in-family','relationship_ Other-relative','relationship_ Own-child','relationship_ Unmarried','relationship_ Wife',
    'capital-gain',
    'capital-loss',
    'hours-per-week',
    'native-country_ Cambodia','native-country_ Canada','native-country_ China','native-country_ Columbia','native-country_ Cuba','native-country_ Dominican-Republic','native-country_ Ecuador','native-country_ El-Salvador','native-country_ England','native-country_ France','native-country_ Germany','native-country_ Greece','native-country_ Guatemala','native-country_ Haiti','native-country_ Holand-Netherlands','native-country_ Honduras','native-country_ Hong','native-country_ Hungary','native-country_ India','native-country_ Iran','native-country_ Ireland','native-country_ Italy','native-country_ Jamaica','native-country_ Japan','native-country_ Laos','native-country_ Mexico','native-country_ Nicaragua','native-country_ Outlying-US(Guam-USVI-etc)','native-country_ Peru','native-country_ Philippines','native-country_ Poland','native-country_ Portugal','native-country_ Puerto-Rico','native-country_ Scotland','native-country_ South','native-country_ Taiwan','native-country_ Thailand','native-country_ Trinadad&Tobago','native-country_ United-States','native-country_ Vietnam','native-country_ Yugoslavia',
    'income'
]

In [43]:
adult_train_data = adult_train_data[columns_order]
adult_test_data = adult_test_data[columns_order]

# Saving data

In [44]:
data = pd.concat([adult_train_data, adult_test_data])

In [45]:
len(data.index)

45222

In [46]:
data.to_csv("../../data/processed/adult/adult_mpa_bin_wout_agg.csv")